In [1]:
from scipy.special import roots_jacobi, eval_jacobi
import pandas as pd
import numpy as np
import uxarray as ux
import xarray as xr

In [166]:
grid_file = ux.open_grid('TEMPEST_ne30.g')
node_connect = grid_file.face_node_connectivity
node_connect[0, :]

<xarray.DataArray 'face_node_connectivity' (n_max_face_nodes: 4)>
array([  0,   8, 356, 124])
Dimensions without coordinates: n_max_face_nodes
Attributes:
    cf_role:      face_node_connectivity
    _FillValue:   -9223372036854775808
    start_index:  0

In [510]:
grid_file.node_x.isel(n_node = grid_file.face_node_connectivity[10]) 

<xarray.DataArray 'node_x' (n_max_face_nodes: 4)>
array([0.69474659, 0.69925278, 0.73404352, 0.72883583])
Coordinates:
    node_lon  (n_max_face_nodes) float64 -15.0 -12.0 -12.0 -15.0
    node_lat  (n_max_face_nodes) float64 -44.01 -44.37 -41.37 -41.01
Dimensions without coordinates: n_max_face_nodes
Attributes:
    standard_name:  x
    long_name:      cartesian x
    units:          m

In [511]:
map_cartesian(grid_file, ref_coords, GLL_points).isel(n_face = 10)['x_comp']

<xarray.DataArray 'x_comp' (x2: 4, y2: 4)>
array([[0.72883583, 0.7303914 , 0.73272072, 0.73404352],
       [0.71961011, 0.72110829, 0.72335167, 0.72462568],
       [0.70436078, 0.70576535, 0.70786835, 0.70906254],
       [0.69474659, 0.69609293, 0.69810845, 0.69925278]])
Coordinates:
    n_face      int64 10
    GLL_points  (x2, y2) int64 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Dimensions without coordinates: x2, y2

In [ ]:
test = find_neighbors(grid_file)

In [ ]:
spherical_coords.isel(n_face = list(test[0][0]))

In [ ]:
import pandas as pd

sr = pd.Series(test[0])
sr.index.name = 'n_face'
sr.to_xarray()

In [4]:
npts = 4
p_order = npts - 1
xinterior, w = roots_jacobi(p_order - 1,1,1) # returns interior GLL nodes from range -1 and 1
GLL_points = np.pad(xinterior, (1, 1), 'constant', constant_values=(-1, 1))

ref_coords = np.meshgrid(GLL_points, GLL_points[::-1])

In [ ]:
node_x = grid_file.node_x.isel(n_node = node_connect)
node_y = grid_file.node_y.isel(n_node = node_connect)
node_z = grid_file.node_z.isel(n_node = node_connect)

In [35]:
t1 = (1 - ref_coords[0]) * (1 - ref_coords[1])
t2 = (1 + ref_coords[0]) * (1 - ref_coords[1])
t3 = (1 + ref_coords[0]) * (1 + ref_coords[1])
t4 = (1 - ref_coords[0]) * (1 + ref_coords[1])

In [169]:
ref_coords

[array([[-1.       , -0.4472136,  0.4472136,  1.       ],
        [-1.       , -0.4472136,  0.4472136,  1.       ],
        [-1.       , -0.4472136,  0.4472136,  1.       ],
        [-1.       , -0.4472136,  0.4472136,  1.       ]]),
 array([[ 1.       ,  1.       ,  1.       ,  1.       ],
        [ 0.4472136,  0.4472136,  0.4472136,  0.4472136],
        [-0.4472136, -0.4472136, -0.4472136, -0.4472136],
        [-1.       , -1.       , -1.       , -1.       ]])]

In [87]:
# Returns Cartesian coordinates on the unit sphere: 
# Fills in starting with C1 from the bottom left, moves counterclockwise:
# GLL_point 12 is c1, GLL_point 15 is c2, GLL_point 3 is c3, GLL_point 0 is c4

def map_cartesian(grid_file, ref_coords, GLL_points):
    
    # Define arrays for corners in x_2, y_2:
    node_connect = grid_file.face_node_connectivity
    
    # Return the x, y, z comps of element corners of faces:
    node_x = grid_file.node_x.isel(n_node = node_connect)
    node_y = grid_file.node_y.isel(n_node = node_connect)
    node_z = grid_file.node_z.isel(n_node = node_connect)
    
    t1 = (1 - ref_coords[0]) * (1 - ref_coords[1])
    t2 = (1 + ref_coords[0]) * (1 - ref_coords[1])
    t3 = (1 + ref_coords[0]) * (1 + ref_coords[1])
    t4 = (1 - ref_coords[0]) * (1 + ref_coords[1])
    
    r1 = ((1/4) * (np.outer(node_x.isel(n_max_face_nodes = 0), t1.reshape(-1))
                   + np.outer(node_x.isel(n_max_face_nodes = 1), t2.reshape(-1)) 
                   + np.outer(node_x.isel(n_max_face_nodes = 2), t3.reshape(-1))
                   + np.outer(node_x.isel(n_max_face_nodes = 3), t4.reshape(-1)))).reshape(5400, 4, 4)
    
    r2 = ((1/4) * (np.outer(node_y.isel(n_max_face_nodes = 0), t1.reshape(-1))
                   + np.outer(node_y.isel(n_max_face_nodes = 1), t2.reshape(-1)) 
                   + np.outer(node_y.isel(n_max_face_nodes = 2), t3.reshape(-1))
                   + np.outer(node_y.isel(n_max_face_nodes = 3), t4.reshape(-1)))).reshape(5400, 4, 4)
    
    r3 = ((1/4) * (np.outer(node_z.isel(n_max_face_nodes = 0), t1.reshape(-1))
                   + np.outer(node_z.isel(n_max_face_nodes = 1), t2.reshape(-1)) 
                   + np.outer(node_z.isel(n_max_face_nodes = 2), t3.reshape(-1))
                   + np.outer(node_z.isel(n_max_face_nodes = 3), t4.reshape(-1)))).reshape(5400, 4, 4)
    
    r_vector = xr.Dataset(data_vars = {'x_comp' : (["n_face", "x2", "y2"], r1), 
                                       'y_comp' : (["n_face", "x2", "y2"], r2), 
                                       'z_comp' : (["n_face", "x2", "y2"], r3)},
                          coords = {'n_face' : (('n_face'), node_x.coords['n_face'].values),
                                    'GLL_points' : (('x2', 'y2'), np.arange(len(GLL_points) ** 2).reshape(len(GLL_points), len(GLL_points)))})
    
    r_norm = r_vector / np.sqrt((r_vector['x_comp'] ** 2) + (r_vector['y_comp'] ** 2) + (r_vector['z_comp'] ** 2))
    return(r_norm)


In [99]:
# Convert Cartesian coordinates on the unit sphere to lat/lon in radians: 

def map_spherical(input_array):
    
    # Note: lon returned from [-pi, pi], lat returned from [0, pi]
    lon_array = np.arctan2(input_array['y_comp'], input_array['x_comp'])
    lat_array = np.arccos(input_array['z_comp'])
    
    # Shift lon to [0, 2pi] and lat to [pi/2, -pi/2]:
    lat_shift = -1 * (lat_array - (np.pi / 2))
    lon_shift = np.where(lon_array < 0, lon_array + (2 * np.pi), lon_array)
    
    return(xr.Dataset(data_vars = {'lat' : (["n_face", "x2", "y2"], lat_shift.data), 
                                   'lon' : (["n_face", "x2", "y2"], lon_shift.data)},
                      coords = {'n_face' : (('n_face'), input_array.coords['n_face'].values), 
                                'GLL_points' : (('x2', 'y2'), input_array.coords['GLL_points'].values)}))


In [62]:
node_connect = grid_file.face_node_connectivity
node_x = grid_file.node_x.isel(n_node = node_connect)
((np.outer(node_x.isel(n_max_face_nodes = 0), t1.reshape(-1))[10] + 
np.outer(node_x.isel(n_max_face_nodes = 1), t2.reshape(-1))[10] +
np.outer(node_x.isel(n_max_face_nodes = 2), t3.reshape(-1))[10] +
np.outer(node_x.isel(n_max_face_nodes = 3), t4.reshape(-1))[10]) * (1/4)).reshape(4, 4)

array([[0.72883583, 0.7302752 , 0.73260415, 0.73404352],
       [0.7194138 , 0.72079958, 0.72304182, 0.7244276 ],
       [0.70416862, 0.70546769, 0.70756963, 0.7088687 ],
       [0.69474659, 0.69599207, 0.6980073 , 0.69925278]])

In [86]:
((np.outer(node_x.isel(n_max_face_nodes = 0), t1.reshape(-1)) + 
np.outer(node_x.isel(n_max_face_nodes = 1), t2.reshape(-1)) +
np.outer(node_x.isel(n_max_face_nodes = 2), t3.reshape(-1)) +
np.outer(node_x.isel(n_max_face_nodes = 3), t4.reshape(-1))) * (1/4)).reshape(5400, 4, 4)

array([[[ 0.59647278,  0.60232066,  0.61178273,  0.6176306 ],
        [ 0.59118745,  0.59687985,  0.60609033,  0.61178273],
        [ 0.5826356 ,  0.58807642,  0.59687985,  0.60232066],
        [ 0.57735027,  0.5826356 ,  0.59118745,  0.59647278]],

       [[ 0.6176306 ,  0.62291202,  0.63145753,  0.63673894],
        [ 0.61178273,  0.61691507,  0.62521937,  0.63035171],
        [ 0.60232066,  0.6072118 ,  0.61512582,  0.62001696],
        [ 0.59647278,  0.60121485,  0.60888766,  0.61362973]],

       [[ 0.63673894,  0.64148692,  0.64916931,  0.65391729],
        [ 0.63035171,  0.63495852,  0.6424125 ,  0.64701931],
        [ 0.62001696,  0.62439535,  0.63147975,  0.63585815],
        [ 0.61362973,  0.61786696,  0.62472294,  0.62896017]],

       ...,

       [[-0.62896017, -0.62472294, -0.61786696, -0.61362973],
        [-0.61785734, -0.61360965, -0.60673673, -0.60248903],
        [-0.59989259, -0.59562796, -0.58872764, -0.58446301],
        [-0.58878977, -0.58451467, -0.57759742, -0.

In [445]:
np.array([(index_array[pair[0]], GLL_index[i_a][-1])], dtype=object)

(1, 2)

In [450]:
xr.DataArray(data = np.array([(index_array[pair[0]], GLL_index[i_a][-1])], dtype=object).reshape(-1), 
             dims = ['n_corners'])

<xarray.DataArray (n_corners: 2)>
array([1, array([3, 0])], dtype=object)
Dimensions without coordinates: n_corners

In [619]:
test = map_cartesian(grid_file, ref_coords, GLL_points)

In [620]:
test['shared_corners'] = xr.DataArray(data = np.zeros((grid_file.n_face, 2, npts, npts), dtype=object), 
                                      dims = ['n_face', 'n_corners','x2', 'y2',])

In [639]:
test.isel(n_face = 1, x2 = 2, n_corners = 1)['shared_corners'].values[:] = test_arr
test.isel(n_face = 1, x2 = 2, n_corners = 0)['shared_corners'].values[:] = np.full((4, ), 14)

test.isel(n_face = 1, x2 = 2)['shared_corners']

<xarray.DataArray 'shared_corners' (n_corners: 2, y2: 4)>
array([[14, 14, 14, 14],
       [(0, 0), (0, 1), (0, 2), (0, 3)]], dtype=object)
Coordinates:
    n_face      int64 1
    GLL_points  (y2) int64 8 9 10 11
Dimensions without coordinates: n_corners, y2

In [640]:
idx_1 = np.array([3])
idx_2 = np.array([0, 3])

test_arr = tuple(zip(np.linspace(idx_1[0], idx_1[-1], 4, dtype = int), 
                     np.linspace(idx_2[0], idx_2[-1], 4, dtype = int)))

test_arr

array([[3, 0],
       [3, 1],
       [3, 2],
       [3, 3]])

In [483]:
# Insert 2 additional data_vars, one for shared edges and one for shared corners:
# These data_vars will have the same dims/coords as input data_vars

def return_neighbors(grid_array, value_array, npts, max_n_corners, max_n_edges):
    
    # Preallocate arrays for corners and edges
    value_array['shared_corners'] = xr.DataArray(data = np.zeros((grid_array.n_face, 2, npts, npts), dtype=object), 
                                                 dims = ['n_face', 'n_corners','x2', 'y2'])
    
    value_array['shared_edges'] = xr.DataArray(data = np.empty((grid_array.n_face, 2, npts, npts), dtype=object), 
                                                 dims = ['n_face', 'n_edges', 'x2', 'y2']) 
    
    for value in np.arange(grid_array.n_node):
        GLL_index = np.array(([npts-1, 0], [npts-1, npts-1], [0, npts-1], [0, 0]))
        index_array = np.where(grid_array.face_node_connectivity == value)[0]
        node_array = grid_array.face_node_connectivity[index_array]
        
        # Create matrix tracking which elements overlap across rows of node_array: 
        shared_counts = np.triu(np.sum(np.sum(np.array([node_array.values[i, :] == node_array.values[:, :, None] for i in np.arange(node_array.shape[0])]),
                                axis = -1), axis = -1))
        
        i, j = np.where(shared_counts == 1)
        shared_corners = list(zip(i, j))
        
        a, b = np.where(shared_counts == 2)
        shared_edges = list(zip(a, b))
            
        # For corners:
        for pair in shared_corners:
            corner_nodes, i_a, j_b = np.intersect1d(node_array[pair[0]], node_array[pair[1]], return_indices=True)
            value_array.isel(n_face = index_array[pair[0]],
                             x2 = GLL_index[i_a][-1][0], y2 = GLL_index[i_a][-1][1])['shared_corners'].values[:] = xr.DataArray(data = np.array([(index_array[pair[1]], GLL_index[j_b][-1])], dtype=object).reshape(-1), 
                                                                                                                                dims = ['n_corners'])
            
            value_array.isel(n_face = index_array[pair[1]], 
                             x2 = GLL_index[j_b][-1][0], y2 = GLL_index[j_b][-1][1])['shared_corners'].values[:] = xr.DataArray(data = np.array([(index_array[pair[0]], GLL_index[i_a][-1])], dtype=object).reshape(-1), 
                                                                                                                                dims = ['n_corners'])
        # For edges - consider repeats:
        for pair in shared_edges:
            edge_nodes, i_a, j_b = np.intersect1d(node_array[pair[0]], node_array[pair[1]], return_indices=True)
            
            # Index edge for one face:
            i_idx_2 = np.unique(np.array([GLL_index[j_b[0]][0], GLL_index[j_b[1]][0]]), sorted = True)
            j_idx_2 = np.unique(np.array([GLL_index[j_b[0]][1], GLL_index[j_b[1]][1]]), sorted = True)
            
            # Index edge for adjacent face:
            i_idx_1 = np.unique(np.array([GLL_index[i_a[0]][0], GLL_index[i_a[1]][0]]), sorted = True)
            j_idx_1 = np.unique(np.array([GLL_index[i_a[0]][1], GLL_index[i_a[1]][1]]), sorted = True)
            
            # Generate point-by-point indices: 
            GLL_idx_2 = tuple(zip(np.linspace(i_idx_2[0], i_idx_2[-1], npts, dtype = int), 
                                  np.linspace(j_idx_2[0], j_idx_2[-1], npts, dtype = int)))
            GLL_idx_1 = tuple(zip(np.linspace(i_idx_1[0], i_idx_1[-1], npts, dtype = int), 
                                  np.linspace(j_idx_1[0], j_idx_1[-1], npts, dtype = int)))
            
            # Write to arrays: 
            value_array.isel(n_face = index_array[pair[1]])['shared_edges'][0][i_idx_2[0] : i_idx_2[-1] + 1, j_idx_2[0] : j_idx_2[-1] + 1].values[:] = np.full((1, npts), index_array[pair[0]])
            value_array.isel(n_face = index_array[pair[1]])['shared_edges'][1][i_idx_2[0] : i_idx_2[-1] + 1, j_idx_2[0] : j_idx_2[-1] + 1].values[:] = GLL_idx_1
            
            value_array.isel(n_face = index_array[pair[0]])['shared_edges'][0][i_idx_1[0] : i_idx_1[-1] + 1, j_idx_1[0] : j_idx_1[-1] + 1].values[:] = np.full((1, npts), index_array[pair[1]])
            value_array.isel(n_face = index_array[pair[0]])['shared_edges'][1][i_idx_1[0] : i_idx_1[-1] + 1, j_idx_1[0] : j_idx_1[-1] + 1].values[:] = GLL_idx_2
            
            
            
    return(value_array)


In [504]:
edge_nodes, i_a, j_b = np.intersect1d(node_array[0], node_array[1], return_indices=True)
np.sort(np.unique(GLL_index[j_b[0]][1], GLL_index[j_b[1]][1]))

array([0])

In [503]:
np.unique(np.array([GLL_index[j_b[0]][0], GLL_index[j_b[1]][0]]))

array([0, 3])

In [485]:
return_neighbors(grid_file, map_cartesian(grid_file, ref_coords, GLL_points), npts, 2, 2)

<xarray.Dataset>
Dimensions:         (n_face: 5400, x2: 4, y2: 4, n_corners: 2, n_edges: 2)
Coordinates:
  * n_face          (n_face) int64 0 1 2 3 4 5 ... 5394 5395 5396 5397 5398 5399
    GLL_points      (x2, y2) int64 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15
Dimensions without coordinates: x2, y2, n_corners, n_edges
Data variables:
    x_comp          (n_face, x2, y2) float64 0.5965 0.6025 ... -0.5425 -0.5371
    y_comp          (n_face, x2, y2) float64 -0.5965 -0.5855 ... 0.5855 0.5965
    z_comp          (n_face, x2, y2) float64 -0.5371 -0.5425 ... 0.6025 0.5965
    shared_corners  (n_face, n_corners, x2, y2) object 2759 0 0 31 ... 0 0 [0 3]
    shared_edges    (n_face, n_edges, x2, y2) object None None ... None None

In [343]:
index_array = np.where(grid_file.face_node_connectivity == 8)[0]
node_array = grid_file.face_node_connectivity[index_array]
index_array

array([   0,    1, 4470, 4471])

In [331]:
shared_counts = np.triu(np.sum(np.sum(np.array([node_array.values[i, :] == node_array.values[:, :, None] for i in np.arange(node_array.shape[0])]),
                               axis = -1), axis = -1))

shared_counts

array([[4, 2, 2],
       [0, 4, 2],
       [0, 0, 4]])

In [205]:
# For edges:
a, b = np.where(shared_counts == 2)
shared_edges = list(zip(a, b))

index_1 = shared_edges[0][0]
index_2 = shared_edges[0][1]

# Return indices of shared edge corners:
edge_nodes, ia, ib = np.intersect1d(node_array[index_1], node_array[index_2], return_indices=True)
ia, ib

(array([1, 2]), array([0, 3]))

In [247]:
# For corners:
i, j = np.where(shared_counts == 1)
shared_corners = list(zip(i, j))

index_1 = shared_corners[0][0]
index_2 = shared_corners[0][1]
corner_nodes, ia, ib = np.intersect1d(node_array[index_1], node_array[index_2], return_indices=True)

for pair in [shared_corners[1]]:
    array_1 = np.empty((npts, npts), dtype=object)
    array_2 = np.empty((npts, npts), dtype=object)
    corner_nodes, i_a, j_b = np.intersect1d(node_array[pair[0]], node_array[pair[1]], return_indices=True)
            
    array_1[GLL_index[i_a][-1][0], GLL_index[i_a][-1][1]] = (index_array[pair[1]], GLL_index[j_b])
    array_2[GLL_index[j_b][-1][0], GLL_index[j_b][-1][1]] = (index_array[pair[0]], GLL_index[i_a])